# 1. ProteinMPNN: design sequences for a fixed backbone

- **Input:** The ubiquitin backbone (1UBQ, chain A; 76 residues).
- **Model information:** Backbone geometry and learned relationships between protein sequences and structures.
- **Design objective:** Generate alternative sequences compatible with the same backbone.

Use ProteinMPNN to sample three sequences, compare them with ubiquitin, and predict the structure of a selected design.


Protein MPNN is trained on structure (not amino acid sequence)

*   Structure --> sequence (not the reverse)
*   Structure is based on distances between Carbons


*   This makes it rotation & translation invariant





## Setup
Select **Runtime → Change runtime type → T4 GPU**, then run the two cells below to load the tools and models.


In [1]:
!git -C BME305-protein-design-tools pull -q 2>/dev/null || git clone -q https://github.com/RomeroLab/BME305-protein-design-tools.git
%pip -q install --upgrade ./BME305-protein-design-tools
from protein_design_tools import load_pdb, load_ESMFold, load_ESM2, load_MPNN, load_RFdiffusion, predict_structure, MPNNdesign, ESM2design, RFdiffusion, alignment, view_structure, inspect_prediction, save_results


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
load_MPNN()
load_ESMFold();


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Some weights of EsmForProteinFolding were not initialized from the model checkpoint at facebook/esmfold_v1 and are newly initialized: ['esm.contact_head.regression.bias', 'esm.contact_head.regression.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Colab has PyMol interface: this is how structure is generated

## 1. Start with a structure
Rotate ubiquitin and find its helices, sheets, and loops. This is the structural constraint for sequence design.


In [3]:
native_seq, reference_pdb = load_pdb('1UBQ', chain='A')
view_structure(reference_pdb)


1UBQ, chain A: 76 residues


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 2. Design sequences
ProteinMPNN samples sequences conditioned on the input backbone. Higher temperature usually produces more diverse sequences. The objective is backbone compatibility rather than recovery of the natural sequence.


In [11]:
designs = MPNNdesign(reference_pdb, num_sequences=5, temperature=5, seed=7)


Model takes backbone, stripped of all side chains, and tries to place amino acids on the backbone

Temperature affects diversity of sampling; can sample less favorable conformations
Higher temperature --> more diversity

Amino Acid selection is based on probability: higher temperatures allow for lower-probability amino acids: At highest temperature, all have P=0.05

Statistically, selected designs are stabilizing

However, selected designs probably don't preserve protein function because catalytic residues may be changed

There are ways to design around catalytic residues


## 3. Compare sequences
Compare each design with ubiquitin and identify positions conserved across the designs. `|` marks a match to ubiquitin; `.` marks a substitution.


In [12]:
alignment({'ubiquitin':native_seq, **designs})



Positions 1–60
ubiquitin    MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYN
design_1     WVIWACCMAGYTPINTLILGWNFTTERQPESMDIGITMAISTHVMQELTLRSLMAKGQVM
             ..|......|.|......................||.............|..........
design_2     DSMWYTSTCEDPQHTAHNGSNFPAWAHVNMCFMLRCKAQCQMCFFIINNSEESQERPKSG
             ...................|....................|...|.....|.........
design_3     KEPFERQNAPQQQPLVKDETQIRAWTYRTSMTIATIILYQREARFENWFPDGSLMADQWA
             ...|..........|....................|...|....|...............
design_4     SAPGVRDEHELCACCHVHFAALRKQWDLSLRKTLHAIDSITKADLMQHTSMDHFFKQWVK
             ....|...........|..................................|........
design_5     THTSLNSEDDADHFQTMLEKCYCTRYRDHVNVVCQGRNVIEKIYYPEWPWMISLPPAYSH
             ............................................................

Positions 61–76
ubiquitin    IQKESTLHLVLRLRGG
design_1     ADDLVRLHFPASPMGR
             ......||......|.
design_2     LLPIVTAFSFDMPREP
             .....|.......|..
des

Align native ubiquitin sequence against three designs

## 4. Predict and compare structures
Predict a selected design with ESMFold and compare it with ubiquitin's experimental backbone. ESMFold uses the sequence alone, without the reference coordinates. RMSD measures backbone agreement; pLDDT describes confidence in the prediction. Inspect the overall fold and regions of disagreement alongside their confidence.


In [13]:
prediction_pdb = predict_structure(designs['design_1'], num_recycles=1)
metrics = inspect_prediction(reference_pdb, prediction_pdb)


Predicted 76 residues: /content/protein_demo/prediction_jrb0nw0_/prediction.pdb
Cα RMSD over all 76 residues: 14.51 Å
Mean Cα pLDDT: 35.5/100


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Gray: reference. Prediction confidence: blue ≥90, cyan 70–90, yellow 50–70, red <50.


Colored:

pLDDT: structure quality (confidence) metric:
Blue: high confidence

alpha-C RMSD: variation in location of alpha carbon compared to pdb structure


## Discuss
- Where does the prediction agree with the backbone, and where does it differ? Check confidence at the mismatches.
- Would a matching fold establish that the design retains ubiquitin's biological function? What else would you test?

## Save results
Download the designed sequences and structures for further analysis.


In [ ]:
bundle = save_results(designs, reference_pdb, prediction_pdb)
from google.colab import files
files.download(str(bundle))


## Reuse the tools
The same tools accept your own sequences and structures. See the [function reference](https://github.com/RomeroLab/BME305-protein-design-tools#function-reference) for use in later assignments.

## References
[ProteinMPNN](https://github.com/dauparas/ProteinMPNN) · [1UBQ](https://www.rcsb.org/structure/1UBQ) · [ESMFold](https://github.com/facebookresearch/esm)
